## Machine Predictions: What Contributes to CHF?
In this notebook, we will show how models can be created from data to make predictions.

### Notebook Setup
Data libraries and visualization:

In [ ]:
# Data Libraries
import pandas
import numpy as np
import matplotlib

# Configure notebook for visualization
%matplotlib inline

Machine learning libraries:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

### Load and Prepare Data
Load Framingham data:

In [ ]:
# Load data remotely
patientdata = pandas.read_csv('https://oak-tree.tech/documents/96/framingham.csv')

Inspect columns and values, determine which to include in the model:

In [ ]:
patientdata.dtypes

Patient data includes observation and outcome columns. Observation columns:

* `male`: gender of the patient (1 for men, 0 for women)
* `age`: age at the time of the most recent checkup
* `education`: highest level of education attained
* `currentSmoker`: does the patient currently smoke
* `cigsPerDay`: how many cigarettes do they smoke
* `BPMeds`: do they take medicine to control blood pressure
* `BMI`: patients current body mass index (measurement of height to weight)
* `sysBP`: systolic blood pressure at most recent checkup
* `diaBP`: diastolic blood pressure at most recent checkup
* `heartRate`: heart rate at most recent checkup
* `totChol`: total cholesterol

Outcome columns:

* `prevalentHyp`: have they been diagnosed with hypertension
* `diabetes`: have they been diagnosed with Diabetes
* `prevalentStroke`: have they had a stroke
* `TenYearCHD`: has the patient developed congestive heart failure (this data point includes at least ten years of follow-up)

_Any of the outcome columns could be used for the model. In this example, we will attempt to predict presence of CHD at ten years._

Fetch columns of interest to a new dataframe:

In [ ]:
chfdata = patientdata[['male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 
	'BPMeds', 'BMI', 'sysBP', 'diaBP', 'heartRate', 'totChol', 
	'prevalentHyp', 'prevalentStroke', 'diabetes', 'TenYearCHD']]
chfdata

Drop observations/rows from the data frame which are not complete (at least one element is missing)

In [ ]:
chfdata = chfdata.dropna(axis='columns')
chfdata

### Create Model
Training models and making predictions (helper methods):

In [ ]:
def train(ModelClass, features, target):
	'''	Train a model using the provided features, target, and model class
	'''
	model = ModelClass()
	model.fit(features, target)
	return model


def predict(model, features):
	'''	Use the provided model to create predictions on the provided features
	'''
	return model.predict(features)

Procedure to sample the testing/training split for the model:

In [ ]:
from sklearn.model_selection import train_test_split


def ml_split_data(target, df, test_size=0.3):
	'''	Create a train/testing split for the specified target from the provided dataframe

		@input test_size (default=0.3): Size of the set which should be used for testing
			the model.
	'''
	# Pull target into a series, drop the target from the data frame
	y = df[target]
	X = df.drop(columns=target)

	return train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)


Apply the mdoel procedure, train the model:

In [ ]:
X_train, X_test, y_train, y_test = ml_split_data('TenYearCHD', chfdata)

Train the model using the training data, test the model using the testing data:

In [ ]:
# Train and assess the model: Decision Tree
dtmodel = train(DecisionTreeClassifier, X_train, y_train)
dtmodel.score(X_test, y_test)

Inspect how important the model thinks the features are:

In [ ]:
# Directly inspect the model for impact
for i, l in enumerate(X_train.columns):
	print(l, dtmodel.feature_importances_[i])

In [ ]:
# Pull the factors from the model and re-order based
# on the importance
factors = [(l, dtmodel.feature_importances_[i])
    for i, l in enumerate(X_train.columns)]
for l, w in sorted(factors, key=lambda v: v[1], reverse=True):
    print(l, w)

### Assess Alternative Models

In [ ]:
rfmodel = train(RandomForestClassifier, X_train, y_train)
rfmodel.score(X_test, y_test)